
# 01. Dataset definitivo de modelagem — contratos depurados

Este notebook usa `datase_depurado_contratos.csv` como **fonte mestre e autoritativa** do universo de modelagem.

As fontes auxiliares são usadas somente para construir features point-in-time:

- `vw_bi_parcelas.csv`
- `vw_tparcela_liberacao.csv`
- `vw_taditivovalor.csv`
- `vw_taditivoprazo.csv`

Nenhum contrato fora da relação depurada entra no dataset.

O target é:

`target_dias_atraso = max(0, data_conclusao_definitiva - data_termino_vigente_final)`

`target_atraso = 1` quando `target_dias_atraso > 0`.

Cada linha do dataset final representa um snapshot mensal de um contrato.


In [1]:

from pathlib import Path
import hashlib
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

FILES = {
    "contratos_depurados": "datase_depurado_contratos.csv",
    "parcelas": "vw_bi_parcelas.csv",
    "liberacoes": "vw_tparcela_liberacao.csv",
    "aditivos_valor": "vw_taditivovalor.csv",
    "aditivos_prazo": "vw_taditivoprazo.csv",
}


## 1. Localização dos arquivos

In [2]:

def find_project_root():
    current = Path.cwd().resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "data" / "raw").exists():
            return candidate

    for candidate in [current, *current.parents]:
        if (candidate / "data" / "processed").exists():
            return candidate

    return current


ROOT = find_project_root()
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
QUALITY = ROOT / "reports" / "quality"

PROCESSED.mkdir(parents=True, exist_ok=True)
QUALITY.mkdir(parents=True, exist_ok=True)


def find_input_file(filename):
    candidates = [
        RAW / filename,
        ROOT / filename,
        Path.cwd() / filename,
        Path("/mnt/data") / filename,
    ]

    for path in candidates:
        if path.exists():
            return path.resolve()

    raise FileNotFoundError(
        f"Arquivo não encontrado: {filename}\n"
        + "\n".join(str(p) for p in candidates)
    )


SOURCE_PATHS = {
    key: find_input_file(filename)
    for key, filename in FILES.items()
}

for key, path in SOURCE_PATHS.items():
    print(f"{key:20s}: {path}")


contratos_depurados : C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras\data\raw\datase_depurado_contratos.csv
parcelas            : C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras\data\raw\vw_bi_parcelas.csv
liberacoes          : C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras\data\raw\vw_tparcela_liberacao.csv
aditivos_valor      : C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras\data\raw\vw_taditivovalor.csv
aditivos_prazo      : C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras\data\raw\vw_taditivoprazo.csv


## 2. Carga e tratamento das fontes

In [3]:

def read_csv(path):
    return pd.read_csv(
        path,
        low_memory=False,
        encoding="utf-8-sig",
    )


def to_dates(df, columns):
    result = df.copy()

    for column in columns:
        if column not in result.columns:
            continue

        result[column] = pd.to_datetime(
            result[column],
            errors="coerce",
        )

        valid = result[column].notna()
        result.loc[
            valid & (result[column].dt.year <= 1900),
            column,
        ] = pd.NaT

    return result


def to_numbers(df, columns):
    result = df.copy()

    for column in columns:
        if column in result.columns:
            result[column] = pd.to_numeric(
                result[column],
                errors="coerce",
            )

    return result


contratos = read_csv(SOURCE_PATHS["contratos_depurados"])
parcelas = read_csv(SOURCE_PATHS["parcelas"])
liberacoes = read_csv(SOURCE_PATHS["liberacoes"])
aditivos_valor = read_csv(SOURCE_PATHS["aditivos_valor"])
aditivos_prazo = read_csv(SOURCE_PATHS["aditivos_prazo"])

required_master = [
    "id_contrato",
    "classificacao_contrato",
    "data_os",
    "data_termino_original",
    "data_termino_vigente_final",
    "data_conclusao_definitiva",
    "status_final_controle",
    "prazo_original_dias",
    "atraso_definitivo_dias",
]

missing = [
    c for c in required_master
    if c not in contratos.columns
]

if missing:
    raise ValueError(
        f"Colunas obrigatórias ausentes na fonte depurada: {missing}"
    )

contratos = to_numbers(
    contratos,
    [
        "id_contrato",
        "prazo_original_dias",
        "atraso_definitivo_dias",
    ],
)

contratos = to_dates(
    contratos,
    [
        "data_os",
        "data_termino_original",
        "data_termino_vigente_final",
        "data_conclusao_definitiva",
    ],
)

contratos["classificacao_contrato"] = (
    contratos["classificacao_contrato"]
    .astype("string")
    .str.strip()
)

if contratos["id_contrato"].isna().any():
    raise ValueError("Há contrato sem id_contrato.")

if contratos["id_contrato"].duplicated().any():
    raise ValueError("Há id_contrato duplicado na fonte depurada.")

contratos["id_contrato"] = contratos["id_contrato"].astype(int)

if contratos[
    [
        "data_os",
        "data_termino_original",
        "data_termino_vigente_final",
        "data_conclusao_definitiva",
        "prazo_original_dias",
        "atraso_definitivo_dias",
    ]
].isna().any().any():
    raise ValueError(
        "A fonte depurada contém nulos em campos essenciais."
    )

# Validação do término original.
termino_original_recalc = (
    contratos["data_os"]
    + pd.to_timedelta(
        contratos["prazo_original_dias"],
        unit="D",
    )
)

if not (
    termino_original_recalc
    == contratos["data_termino_original"]
).all():
    raise ValueError(
        "data_termino_original diverge de data_os + prazo_original_dias."
    )

# O campo textual da planilha não é usado. A diferença é recalculada.
contratos["diferenca_vigente_original_final_controle"] = (
    contratos["data_termino_vigente_final"]
    - contratos["data_termino_original"]
).dt.days

contratos["target_dias_atraso_recalculado"] = (
    contratos["data_conclusao_definitiva"]
    - contratos["data_termino_vigente_final"]
).dt.days.clip(lower=0)

if not np.allclose(
    contratos["target_dias_atraso_recalculado"].astype(float),
    contratos["atraso_definitivo_dias"].astype(float),
):
    raise ValueError(
        "O target recalculado diverge de atraso_definitivo_dias."
    )

# Fontes auxiliares.
parcelas = to_numbers(
    parcelas,
    [
        "id_parcela",
        "id_contrato",
        "valor_reajustado",
        "ordem_status",
        "valor_ja_liberado",
    ],
)
parcelas = to_dates(
    parcelas,
    [
        "data_prevista_parcela",
        "data_efetiva",
        "data_liberacao",
    ],
)

liberacoes = to_numbers(
    liberacoes,
    [
        "id_Contrato",
        "id_Parcela",
        "Abono_atraso",
        "Atraso_considerado",
        "Valor_total_liberado",
    ],
)
liberacoes = to_dates(
    liberacoes,
    [
        "Data_efet_conclusao",
        "Data_liberação",
    ],
)

aditivos_valor = to_numbers(
    aditivos_valor,
    ["idContrato"],
)
aditivos_valor = to_dates(
    aditivos_valor,
    ["Data_assinatura_TA_valor"],
)

aditivos_prazo = to_numbers(
    aditivos_prazo,
    [
        "idContrato",
        "Aditado_em_dias",
    ],
)
aditivos_prazo = to_dates(
    aditivos_prazo,
    [
        "Data_termino_anterior",
        "Data_termino_nova",
        "Data_assinatura_ta_prazo",
    ],
)

ids_modelagem = set(
    contratos["id_contrato"].astype(int)
)

parcelas = parcelas[
    parcelas["id_contrato"].isin(ids_modelagem)
].copy()

liberacoes = liberacoes[
    liberacoes["id_Contrato"].isin(ids_modelagem)
].copy()

aditivos_valor = aditivos_valor[
    aditivos_valor["idContrato"].isin(ids_modelagem)
].copy()

aditivos_prazo = aditivos_prazo[
    aditivos_prazo["idContrato"].isin(ids_modelagem)
].copy()

coverage = pd.DataFrame([
    {
        "fonte": "contratos_depurados",
        "linhas": len(contratos),
        "contratos": contratos["id_contrato"].nunique(),
    },
    {
        "fonte": "parcelas",
        "linhas": len(parcelas),
        "contratos": parcelas["id_contrato"].nunique(),
    },
    {
        "fonte": "liberacoes",
        "linhas": len(liberacoes),
        "contratos": liberacoes["id_Contrato"].nunique(),
    },
    {
        "fonte": "aditivos_valor",
        "linhas": len(aditivos_valor),
        "contratos": aditivos_valor["idContrato"].nunique(),
    },
    {
        "fonte": "aditivos_prazo",
        "linhas": len(aditivos_prazo),
        "contratos": aditivos_prazo["idContrato"].nunique(),
    },
])

display(coverage)

print("Contratos depurados:", len(contratos))
print(
    "Com atraso:",
    int((contratos["atraso_definitivo_dias"] > 0).sum()),
)
print(
    "Sem atraso:",
    int((contratos["atraso_definitivo_dias"] == 0).sum()),
)


,fonte,linhas,contratos
0,contratos_depurados,45,45
1,parcelas,856,45
2,liberacoes,845,45
3,aditivos_valor,128,36
4,aditivos_prazo,81,36


Contratos depurados: 45
Com atraso: 17
Sem atraso: 28



## 3. Validação dos aditivos de prazo

O último prazo reconstruído com `vw_taditivoprazo` deve coincidir com `data_termino_vigente_final` da fonte depurada.


In [4]:

validacao_aditivos = []

for _, contrato in contratos.iterrows():
    cid = int(contrato["id_contrato"])

    ap = aditivos_prazo[
        aditivos_prazo["idContrato"].eq(cid)
        & aditivos_prazo["Data_assinatura_ta_prazo"].notna()
        & aditivos_prazo["Data_termino_nova"].notna()
        & (
            aditivos_prazo["Data_assinatura_ta_prazo"]
            <= contrato["data_conclusao_definitiva"]
        )
    ].sort_values(
        [
            "Data_assinatura_ta_prazo",
            "Data_termino_nova",
        ]
    )

    prazo_reconstruido = (
        ap["Data_termino_nova"].iloc[-1]
        if len(ap)
        else contrato["data_termino_original"]
    )

    validacao_aditivos.append({
        "id_contrato": cid,
        "prazo_auditado": contrato[
            "data_termino_vigente_final"
        ],
        "prazo_reconstruido": prazo_reconstruido,
        "qtd_aditivos": len(ap),
    })

validacao_aditivos = pd.DataFrame(validacao_aditivos)

validacao_aditivos["diferenca_dias"] = (
    validacao_aditivos["prazo_auditado"]
    - validacao_aditivos["prazo_reconstruido"]
).dt.days

if not validacao_aditivos["diferenca_dias"].eq(0).all():
    display(
        validacao_aditivos[
            ~validacao_aditivos["diferenca_dias"].eq(0)
        ]
    )
    raise ValueError(
        "O prazo final reconstruído diverge da fonte depurada."
    )

print(
    "Prazo final reconstruído coincide com a fonte depurada "
    "para todos os contratos."
)


Prazo final reconstruído coincide com a fonte depurada para todos os contratos.


## 4. Calendário mensal de snapshots

In [5]:

calendar_rows = []

for _, contrato in contratos.iterrows():
    cid = int(contrato["id_contrato"])
    inicio = contrato["data_os"]
    fim = contrato["data_conclusao_definitiva"]

    if fim < inicio:
        raise ValueError(
            f"Contrato {cid}: conclusão anterior à data_os."
        )

    cortes = list(
        pd.date_range(
            inicio,
            fim,
            freq="ME",
        )
    )

    if not cortes or cortes[0] > inicio:
        cortes = [inicio] + cortes

    if cortes[-1] != fim:
        cortes.append(fim)

    for data_corte in sorted(set(cortes)):
        calendar_rows.append(
            (cid, pd.Timestamp(data_corte))
        )

calendar = pd.DataFrame(
    calendar_rows,
    columns=[
        "id_contrato",
        "data_corte",
    ],
)

if calendar.duplicated(
    [
        "id_contrato",
        "data_corte",
    ]
).any():
    raise ValueError(
        "Foram encontrados snapshots duplicados."
    )

print("Snapshots:", len(calendar))
print("Contratos:", calendar["id_contrato"].nunique())
print("Período:", calendar["data_corte"].min(), "a", calendar["data_corte"].max())


Snapshots: 587
Contratos: 45
Período: 2018-10-08 00:00:00 a 2026-02-10 00:00:00



## 5. Features point-in-time

Nenhuma informação futura à `data_corte` é usada como feature.

`data_conclusao_definitiva` e `data_termino_vigente_final` são mantidas apenas como controles para auditoria e target.


In [6]:

contratos_idx = contratos.set_index(
    "id_contrato",
    drop=False,
)

parcelas_por_contrato = {
    int(k): g.copy()
    for k, g in parcelas.groupby("id_contrato")
    if pd.notna(k)
}

liberacoes_por_contrato = {
    int(k): g.copy()
    for k, g in liberacoes.groupby("id_Contrato")
    if pd.notna(k)
}

aditivos_valor_por_contrato = {
    int(k): g.copy()
    for k, g in aditivos_valor.groupby("idContrato")
    if pd.notna(k)
}

aditivos_prazo_por_contrato = {
    int(k): g.copy()
    for k, g in aditivos_prazo.groupby("idContrato")
    if pd.notna(k)
}

empty_parcelas = parcelas.iloc[0:0].copy()
empty_liberacoes = liberacoes.iloc[0:0].copy()
empty_aditivos_valor = aditivos_valor.iloc[0:0].copy()
empty_aditivos_prazo = aditivos_prazo.iloc[0:0].copy()


def parcel_window_features(
    concluidas,
    data_corte,
    days,
):
    window = concluidas[
        concluidas["data_efetiva"]
        >= data_corte - pd.Timedelta(days=days)
    ]

    delays = (
        window["data_efetiva"]
        - window["data_prevista_parcela"]
    ).dt.days.dropna()

    return {
        f"recent_qtd_parcelas_concluidas_{days}d": len(window),
        f"recent_qtd_parcelas_atrasadas_{days}d": (
            int((delays > 0).sum())
            if len(delays)
            else 0
        ),
        f"recent_pct_parcelas_atrasadas_{days}d": (
            float((delays > 0).mean())
            if len(delays)
            else np.nan
        ),
        f"recent_atraso_medio_parcelas_{days}d": (
            float(delays.mean())
            if len(delays)
            else np.nan
        ),
        f"recent_atraso_maximo_parcelas_{days}d": (
            float(delays.max())
            if len(delays)
            else np.nan
        ),
    }


def release_window_features(
    released,
    data_corte,
    days,
):
    window = released[
        released["Data_liberação"]
        >= data_corte - pd.Timedelta(days=days)
    ]

    return {
        f"recent_qtd_liberacoes_{days}d": len(window),
        f"recent_valor_liberado_{days}d": (
            float(
                window[
                    "Valor_total_liberado"
                ].sum(min_count=1)
            )
            if len(window)
            else 0.0
        ),
        f"recent_atraso_medio_liberacao_{days}d": (
            float(
                window[
                    "Atraso_considerado"
                ].mean()
            )
            if len(window)
            else np.nan
        ),
    }


rows = []

for cid, data_corte in calendar.itertuples(
    index=False,
    name=None,
):
    contrato = contratos_idx.loc[cid]

    data_os = contrato["data_os"]
    data_termino_original = contrato[
        "data_termino_original"
    ]

    ap_all = aditivos_prazo_por_contrato.get(
        cid,
        empty_aditivos_prazo,
    )

    ap = ap_all[
        ap_all["Data_assinatura_ta_prazo"].notna()
        & ap_all["Data_termino_nova"].notna()
        & (
            ap_all["Data_assinatura_ta_prazo"]
            <= data_corte
        )
    ].sort_values(
        [
            "Data_assinatura_ta_prazo",
            "Data_termino_nova",
        ]
    )

    data_termino_vigente_corte = (
        ap["Data_termino_nova"].iloc[-1]
        if len(ap)
        else data_termino_original
    )

    pi = parcelas_por_contrato.get(
        cid,
        empty_parcelas,
    )

    parcelas_previstas = pi[
        pi["data_prevista_parcela"].notna()
        & (
            pi["data_prevista_parcela"]
            <= data_corte
        )
    ]

    parcelas_concluidas = pi[
        pi["data_efetiva"].notna()
        & (
            pi["data_efetiva"]
            <= data_corte
        )
    ]

    atrasos = (
        parcelas_concluidas["data_efetiva"]
        - parcelas_concluidas["data_prevista_parcela"]
    ).dt.days.dropna()

    li_all = liberacoes_por_contrato.get(
        cid,
        empty_liberacoes,
    )

    li = li_all[
        li_all["Data_liberação"].notna()
        & (
            li_all["Data_liberação"]
            <= data_corte
        )
    ]

    av_all = aditivos_valor_por_contrato.get(
        cid,
        empty_aditivos_valor,
    )

    av = av_all[
        av_all["Data_assinatura_TA_valor"].notna()
        & (
            av_all["Data_assinatura_TA_valor"]
            <= data_corte
        )
    ]

    dias_desde_os = (
        data_corte - data_os
    ).days

    dias_restantes_prazo_vigente = (
        data_termino_vigente_corte
        - data_corte
    ).days

    prazo_vigente_total_dias = (
        data_termino_vigente_corte
        - data_os
    ).days

    pct_prazo_consumido = (
        dias_desde_os / prazo_vigente_total_dias
        if prazo_vigente_total_dias > 0
        else np.nan
    )

    pct_prazo_restante = (
        dias_restantes_prazo_vigente
        / prazo_vigente_total_dias
        if prazo_vigente_total_dias > 0
        else np.nan
    )

    qtd_previstas = len(parcelas_previstas)
    qtd_concluidas = len(parcelas_concluidas)

    pct_parcelas_concluidas = (
        qtd_concluidas / qtd_previstas
        if qtd_previstas
        else np.nan
    )

    recent = {}
    for days in [30, 60, 90]:
        recent.update(
            parcel_window_features(
                parcelas_concluidas,
                data_corte,
                days,
            )
        )

        recent.update(
            release_window_features(
                li,
                data_corte,
                days,
            )
        )

    atraso_medio_hist = (
        float(atrasos.mean())
        if len(atrasos)
        else np.nan
    )

    atraso_medio_90d = recent[
        "recent_atraso_medio_parcelas_90d"
    ]

    delta_atraso_90d_vs_historico = (
        atraso_medio_90d - atraso_medio_hist
        if (
            pd.notna(atraso_medio_90d)
            and pd.notna(atraso_medio_hist)
        )
        else np.nan
    )

    dias_desde_ultima_parcela = (
        (
            data_corte
            - parcelas_concluidas[
                "data_efetiva"
            ].max()
        ).days
        if len(parcelas_concluidas)
        else np.nan
    )

    dias_desde_ultima_liberacao = (
        (
            data_corte
            - li["Data_liberação"].max()
        ).days
        if len(li)
        else np.nan
    )

    gap_prazo_execucao = (
        pct_prazo_consumido
        - pct_parcelas_concluidas
        if (
            pd.notna(pct_prazo_consumido)
            and pd.notna(pct_parcelas_concluidas)
        )
        else np.nan
    )

    pressao_tempo_execucao = (
        pct_prazo_consumido
        / pct_parcelas_concluidas
        if (
            pd.notna(pct_prazo_consumido)
            and pd.notna(pct_parcelas_concluidas)
            and pct_parcelas_concluidas > 0.01
        )
        else np.nan
    )

    dias_aditados_acumulados = (
        float(
            ap["Aditado_em_dias"].sum(min_count=1)
        )
        if len(ap)
        else 0.0
    )

    pct_dias_aditados = (
        dias_aditados_acumulados
        / contrato["prazo_original_dias"]
        if contrato["prazo_original_dias"] > 0
        else np.nan
    )

    target_dias_atraso = float(
        contrato["atraso_definitivo_dias"]
    )

    row = {
        "id_contrato": cid,
        "data_corte": data_corte,
        "classificacao_contrato": contrato[
            "classificacao_contrato"
        ],
        "data_os": data_os,
        "prazo_original_dias": contrato[
            "prazo_original_dias"
        ],
        "data_termino_original": (
            data_termino_original
        ),
        "data_termino_vigente_corte": (
            data_termino_vigente_corte
        ),
        "dias_desde_os": dias_desde_os,
        "dias_restantes_prazo_vigente": (
            dias_restantes_prazo_vigente
        ),
        "prazo_vigente_total_dias": (
            prazo_vigente_total_dias
        ),
        "pct_prazo_consumido": (
            pct_prazo_consumido
        ),
        "pct_prazo_restante": (
            pct_prazo_restante
        ),
        "qtd_aditivos_prazo": len(ap),
        "dias_aditados_acumulados": (
            dias_aditados_acumulados
        ),
        "pct_dias_aditados_sobre_prazo_original": (
            pct_dias_aditados
        ),
        "dias_desde_ultimo_aditivo_prazo": (
            (
                data_corte
                - ap[
                    "Data_assinatura_ta_prazo"
                ].max()
            ).days
            if len(ap)
            else np.nan
        ),
        "qtd_aditivos_valor": len(av),
        "dias_desde_ultimo_aditivo_valor": (
            (
                data_corte
                - av[
                    "Data_assinatura_TA_valor"
                ].max()
            ).days
            if len(av)
            else np.nan
        ),
        "qtd_parcelas_previstas_ate_corte": (
            qtd_previstas
        ),
        "qtd_parcelas_concluidas_ate_corte": (
            qtd_concluidas
        ),
        "pct_parcelas_concluidas_vs_previstas": (
            pct_parcelas_concluidas
        ),
        "atraso_medio_parcelas_dias": (
            atraso_medio_hist
        ),
        "atraso_mediano_parcelas_dias": (
            float(atrasos.median())
            if len(atrasos)
            else np.nan
        ),
        "atraso_maximo_parcelas_dias": (
            float(atrasos.max())
            if len(atrasos)
            else np.nan
        ),
        "qtd_parcelas_atrasadas": (
            int((atrasos > 0).sum())
            if len(atrasos)
            else 0
        ),
        "pct_parcelas_atrasadas": (
            float((atrasos > 0).mean())
            if len(atrasos)
            else np.nan
        ),
        "dias_desde_ultima_parcela_concluida": (
            dias_desde_ultima_parcela
        ),
        "delta_atraso_90d_vs_historico": (
            delta_atraso_90d_vs_historico
        ),
        "qtd_liberacoes": len(li),
        "valor_liberado_acumulado": (
            float(
                li[
                    "Valor_total_liberado"
                ].sum(min_count=1)
            )
            if len(li)
            else 0.0
        ),
        "atraso_medio_liberacao_dias": (
            float(
                li[
                    "Atraso_considerado"
                ].mean()
            )
            if len(li)
            else np.nan
        ),
        "atraso_maximo_liberacao_dias": (
            float(
                li[
                    "Atraso_considerado"
                ].max()
            )
            if len(li)
            else np.nan
        ),
        "dias_desde_ultima_liberacao": (
            dias_desde_ultima_liberacao
        ),
        "gap_prazo_execucao": (
            gap_prazo_execucao
        ),
        "pressao_tempo_execucao": (
            pressao_tempo_execucao
        ),
        "teve_parcela_concluida": int(
            len(parcelas_concluidas) > 0
        ),
        "teve_liberacao": int(
            len(li) > 0
        ),
        "teve_aditivo_prazo": int(
            len(ap) > 0
        ),
        "teve_aditivo_valor": int(
            len(av) > 0
        ),

        # Campos de controle / leakage.
        "status_final_controle": contrato[
            "status_final_controle"
        ],
        "data_conclusao_definitiva": contrato[
            "data_conclusao_definitiva"
        ],
        "data_conclusao_controle": contrato[
            "data_conclusao_definitiva"
        ],
        "data_termino_vigente_final_controle": contrato[
            "data_termino_vigente_final"
        ],
        "diferenca_vigente_original_final_controle": contrato[
            "diferenca_vigente_original_final_controle"
        ],
        "target_dias_atraso": (
            target_dias_atraso
        ),
        "target_atraso": int(
            target_dias_atraso > 0
        ),
    }

    row.update(recent)
    rows.append(row)

dataset = pd.DataFrame(rows)

dataset["ano_corte"] = dataset["data_corte"].dt.year
dataset["mes_corte"] = dataset["data_corte"].dt.month
dataset["mes_sin"] = np.sin(
    2 * np.pi * dataset["mes_corte"] / 12
)
dataset["mes_cos"] = np.cos(
    2 * np.pi * dataset["mes_corte"] / 12
)

print("Shape:", dataset.shape)
display(dataset.head())


Shape: (587, 74)


,id_contrato,data_corte,classificacao_contrato,data_os,prazo_original_dias,data_termino_original,data_termino_vigente_corte,dias_desde_os,dias_restantes_prazo_vigente,prazo_vigente_total_dias,pct_prazo_consumido,pct_prazo_restante,qtd_aditivos_prazo,dias_aditados_acumulados,pct_dias_aditados_sobre_prazo_original,dias_desde_ultimo_aditivo_prazo,qtd_aditivos_valor,dias_desde_ultimo_aditivo_valor,qtd_parcelas_previstas_ate_corte,qtd_parcelas_concluidas_ate_corte,pct_parcelas_concluidas_vs_previstas,atraso_medio_parcelas_dias,atraso_mediano_parcelas_dias,atraso_maximo_parcelas_dias,qtd_parcelas_atrasadas,pct_parcelas_atrasadas,dias_desde_ultima_parcela_concluida,delta_atraso_90d_vs_historico,qtd_liberacoes,valor_liberado_acumulado,atraso_medio_liberacao_dias,atraso_maximo_liberacao_dias,dias_desde_ultima_liberacao,gap_prazo_execucao,pressao_tempo_execucao,teve_parcela_concluida,teve_liberacao,teve_aditivo_prazo,teve_aditivo_valor,status_final_controle,data_conclusao_definitiva,data_conclusao_controle,data_termino_vigente_final_controle,diferenca_vigente_original_final_controle,target_dias_atraso,target_atraso,recent_qtd_parcelas_concluidas_30d,recent_qtd_parcelas_atrasadas_30d,recent_pct_parcelas_atrasadas_30d,recent_atraso_medio_parcelas_30d,recent_atraso_maximo_parcelas_30d,recent_qtd_liberacoes_30d,recent_valor_liberado_30d,recent_atraso_medio_liberacao_30d,recent_qtd_parcelas_concluidas_60d,recent_qtd_parcelas_atrasadas_60d,recent_pct_parcelas_atrasadas_60d,recent_atraso_medio_parcelas_60d,recent_atraso_maximo_parcelas_60d,recent_qtd_liberacoes_60d,recent_valor_liberado_60d,recent_atraso_medio_liberacao_60d,recent_qtd_parcelas_concluidas_90d,recent_qtd_parcelas_atrasadas_90d,recent_pct_parcelas_atrasadas_90d,recent_atraso_medio_parcelas_90d,recent_atraso_maximo_parcelas_90d,recent_qtd_liberacoes_90d,recent_valor_liberado_90d,recent_atraso_medio_liberacao_90d,ano_corte,mes_corte,mes_sin,mes_cos
0,11,2019-07-24,Reforma Parcial,2019-07-24,150,2019-12-21,2019-12-21,0,150,150,0.0000,1.0000,0,0.0000,0.0000,NaN,1,23.0000,0,0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0,0.0000,NaN,NaN,NaN,NaN,NaN,0,0,0,1,CONCLUÍDO,2020-01-15,2020-01-15,2019-12-21,0,25.0000,1,0,0,NaN,NaN,NaN,0,0.0000,NaN,0,0,NaN,NaN,NaN,0,0.0000,NaN,0,0,NaN,NaN,NaN,0,0.0000,NaN,2019,7,-0.5000,-0.8660
1,11,2019-07-31,Reforma Parcial,2019-07-24,150,2019-12-21,2019-12-21,7,143,150,0.0467,0.9533,0,0.0000,0.0000,NaN,1,30.0000,0,0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0,0.0000,NaN,NaN,NaN,NaN,NaN,0,0,0,1,CONCLUÍDO,2020-01-15,2020-01-15,2019-12-21,0,25.0000,1,0,0,NaN,NaN,NaN,0,0.0000,NaN,0,0,NaN,NaN,NaN,0,0.0000,NaN,0,0,NaN,NaN,NaN,0,0.0000,NaN,2019,7,-0.5000,-0.8660
2,11,2019-08-31,Reforma Parcial,2019-07-24,150,2019-12-21,2019-12-21,38,112,150,0.2533,0.7467,0,0.0000,0.0000,NaN,1,61.0000,1,1,1.0000,-3.0000,-3.0000,-3.0000,0,0.0000,11.0000,0.0000,1,"19,716.1800",0.0000,0.0000,3.0000,-0.7467,0.2533,1,1,0,1,CONCLUÍDO,2020-01-15,2020-01-15,2019-12-21,0,25.0000,1,1,0,0.0000,-3.0000,-3.0000,1,"19,716.1800",0.0000,1,0,0.0000,-3.0000,-3.0000,1,"19,716.1800",0.0000,1,0,0.0000,-3.0000,-3.0000,1,"19,716.1800",0.0000,2019,8,-0.8660,-0.5000
3,11,2019-09-30,Reforma Parcial,2019-07-24,150,2019-12-21,2019-12-21,68,82,150,0.4533,0.5467,0,0.0000,0.0000,NaN,1,91.0000,2,3,1.5000,-14.3333,-12.0000,-3.0000,0,0.0000,6.0000,0.0000,3,"59,228.5600",0.0000,0.0000,0.0000,-1.0467,0.3022,1,1,0,1,CONCLUÍDO,2020-01-15,2020-01-15,2019-12-21,0,25.0000,1,2,0,0.0000,-20.0000,-12.0000,2,"39,512.3800",0.0000,3,0,0.0000,-14.3333,-3.0000,3,"59,228.5600",0.0000,3,0,0.0000,-14.3333,-3.0000,3,"59,228.5600",0.0000,2019,9,-1.0000,-0.0000
4,11,2019-10-31,Reforma Parcial,2019-07-24,150,2019-12-21,2019-12-21,99,51,150,0.6600,0.3400,0,0.0000,0.0000,NaN,1,122.0000,3,3,1.0000,-14.3333,-12.0000,-3.0000,0,0.0000,37.0000,0.0000,3,"59,228.5600",0.0000,0.0000,31.0000,-0.3400,0.6600,1,1,0,1,CONCLUÍDO,2020-01-15,2020-01-15,2019-12-21,0,25.0000,1,0,0,NaN,NaN,NaN,0,0.0000,NaN,2,0,0.0000,-20.0000,-12.0000,2,"39,512.3800",0.0000,3,0,0.0000,-14.3333,-3.0000

## 6. Elegibilidade, pesos e split temporal

In [7]:

dataset["faixa_antecedencia"] = pd.cut(
    dataset["dias_restantes_prazo_vigente"],
    [
        -np.inf,
        29,
        89,
        179,
        np.inf,
    ],
    labels=[
        "<30d",
        "30-89d",
        "90-179d",
        ">=180d",
    ],
)

dataset["flag_apto_treino"] = (
    dataset["data_corte"]
    < dataset["data_conclusao_definitiva"]
).astype("int8")

# Peso igual por contrato.
counts = (
    dataset[
        dataset["flag_apto_treino"].eq(1)
    ]
    .groupby("id_contrato")
    .size()
)

dataset["sample_weight_contrato_raw"] = (
    dataset["id_contrato"].map(
        lambda cid: (
            1.0 / counts[cid]
            if cid in counts.index
            else np.nan
        )
    )
)

mean_w = dataset.loc[
    dataset["flag_apto_treino"].eq(1),
    "sample_weight_contrato_raw",
].mean()

dataset["sample_weight_contrato"] = (
    dataset["sample_weight_contrato_raw"]
    / mean_w
)

# Split temporal por contrato: 70% / 15% / restante.
ordered = (
    contratos[
        [
            "id_contrato",
            "data_conclusao_definitiva",
        ]
    ]
    .sort_values(
        [
            "data_conclusao_definitiva",
            "id_contrato",
        ]
    )
    .reset_index(drop=True)
)

n = len(ordered)
n_train = int(np.floor(n * 0.70))
n_val = int(np.floor(n * 0.15))

ids_train = set(
    ordered.iloc[:n_train]["id_contrato"]
)

ids_val = set(
    ordered.iloc[
        n_train:n_train + n_val
    ]["id_contrato"]
)

ids_test = set(
    ordered.iloc[
        n_train + n_val:
    ]["id_contrato"]
)

split_map = {
    **{cid: "train" for cid in ids_train},
    **{cid: "validation" for cid in ids_val},
    **{cid: "test" for cid in ids_test},
}

dataset["split_contrato_temporal"] = (
    dataset["id_contrato"].map(split_map)
)

dataset["split_modelagem"] = np.where(
    dataset["flag_apto_treino"].eq(1),
    dataset["split_contrato_temporal"],
    "nao_supervisionado",
)

dataset = dataset.sort_values(
    [
        "id_contrato",
        "data_corte",
    ]
).reset_index(drop=True)

split_summary = (
    dataset[
        dataset["flag_apto_treino"].eq(1)
    ]
    .groupby("split_modelagem")
    .agg(
        snapshots=("id_contrato", "size"),
        contratos=("id_contrato", "nunique"),
        taxa_atraso=("target_atraso", "mean"),
        mediana_atraso_dias=(
            "target_dias_atraso",
            "median",
        ),
    )
)

display(split_summary)


,snapshots,contratos,taxa_atraso,mediana_atraso_dias
split_modelagem,,,,
test,149,8,0.1745,0.0000
train,333,31,0.4444,0.0000
validation,60,6,0.5667,14.0000


## 7. Validações finais

In [8]:

# Universo idêntico à fonte depurada.
assert set(
    dataset["id_contrato"].unique()
) == ids_modelagem

# Snapshot único.
assert not dataset.duplicated(
    [
        "id_contrato",
        "data_corte",
    ]
).any()

# Snapshot supervisionado é anterior à conclusão.
train_mask = dataset[
    "flag_apto_treino"
].eq(1)

assert (
    dataset.loc[
        train_mask,
        "data_corte",
    ]
    <
    dataset.loc[
        train_mask,
        "data_conclusao_definitiva",
    ]
).all()

# Target binário.
assert (
    (
        dataset["target_dias_atraso"] > 0
    ).astype(int).values
    ==
    dataset["target_atraso"].astype(int).values
).all()

# Target em dias.
target_recalc = (
    dataset["data_conclusao_definitiva"]
    - dataset[
        "data_termino_vigente_final_controle"
    ]
).dt.days.clip(lower=0)

assert np.allclose(
    target_recalc.astype(float),
    dataset["target_dias_atraso"].astype(float),
)

# Contrato não pode aparecer em mais de um split.
assert (
    dataset.groupby(
        "id_contrato"
    )["split_contrato_temporal"].nunique().max()
    == 1
)

# Peso total equivalente entre contratos.
weight_totals = (
    dataset[
        dataset["flag_apto_treino"].eq(1)
    ]
    .groupby("id_contrato")[
        "sample_weight_contrato"
    ]
    .sum()
)

assert np.allclose(
    weight_totals.values,
    weight_totals.iloc[0],
)

print("Todas as validações foram concluídas com sucesso.")


Todas as validações foram concluídas com sucesso.



## 8. Exportação

O notebook grava:

- `dataset_modelagem_obras_snapshots_definitivo.csv`
- `dataset_modelagem_obras_snapshots.csv` como nome canônico para compatibilidade
- `dicionario_dataset_modelagem_definitivo.csv`


In [9]:

OUTPUT_DEFINITIVO = (
    PROCESSED
    / "dataset_modelagem_obras_snapshots_definitivo.csv"
)

OUTPUT_CANONICO = (
    PROCESSED
    / "dataset_modelagem_obras_snapshots.csv"
)

dataset.to_csv(
    OUTPUT_DEFINITIVO,
    index=False,
    encoding="utf-8-sig",
)

dataset.to_csv(
    OUTPUT_CANONICO,
    index=False,
    encoding="utf-8-sig",
)

CONTROL_COLUMNS = {
    "id_contrato",
    "data_corte",
    "status_final_controle",
    "data_conclusao_definitiva",
    "data_conclusao_controle",
    "data_termino_vigente_final_controle",
    "diferenca_vigente_original_final_controle",
    "target_dias_atraso",
    "target_atraso",
    "flag_apto_treino",
    "sample_weight_contrato_raw",
    "sample_weight_contrato",
    "split_contrato_temporal",
    "split_modelagem",
}

TARGET_COLUMNS = {
    "target_dias_atraso",
    "target_atraso",
}

dictionary_rows = []

for column in dataset.columns:
    if column in TARGET_COLUMNS:
        role = "target"
    elif column in CONTROL_COLUMNS:
        role = "controle"
    else:
        role = "feature_candidata"

    dictionary_rows.append({
        "coluna": column,
        "papel": role,
        "dtype": str(dataset[column].dtype),
        "nulos_pct": round(
            dataset[column].isna().mean() * 100,
            2,
        ),
    })

dictionary = pd.DataFrame(dictionary_rows)

DICTIONARY_PATH = (
    PROCESSED
    / "dicionario_dataset_modelagem_definitivo.csv"
)

dictionary.to_csv(
    DICTIONARY_PATH,
    index=False,
    encoding="utf-8-sig",
)

quality = pd.DataFrame({
    "metrica": [
        "contratos_depurados",
        "snapshots",
        "snapshots_aptos_treino",
        "contratos_aptos_treino",
        "contratos_com_atraso",
        "contratos_sem_atraso",
        "taxa_atraso_contratos",
        "mediana_target_dias_contratos",
        "media_target_dias_contratos",
        "data_conclusao_min",
        "data_conclusao_max",
        "duplicidades_id_data",
    ],
    "valor": [
        contratos["id_contrato"].nunique(),
        len(dataset),
        int(dataset["flag_apto_treino"].sum()),
        dataset.loc[
            dataset["flag_apto_treino"].eq(1),
            "id_contrato",
        ].nunique(),
        int(
            (
                contratos["atraso_definitivo_dias"] > 0
            ).sum()
        ),
        int(
            (
                contratos["atraso_definitivo_dias"] == 0
            ).sum()
        ),
        float(
            (
                contratos["atraso_definitivo_dias"] > 0
            ).mean()
        ),
        float(
            contratos["atraso_definitivo_dias"].median()
        ),
        float(
            contratos["atraso_definitivo_dias"].mean()
        ),
        contratos["data_conclusao_definitiva"].min(),
        contratos["data_conclusao_definitiva"].max(),
        int(
            dataset.duplicated(
                [
                    "id_contrato",
                    "data_corte",
                ]
            ).sum()
        ),
    ],
})

QUALITY_PATH = (
    QUALITY
    / "qualidade_dataset_modelagem_definitivo.csv"
)

quality.to_csv(
    QUALITY_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("=" * 72)
print("DATASET DEFINITIVO GERADO")
print("=" * 72)
print("Arquivo definitivo:", OUTPUT_DEFINITIVO.resolve())
print("Arquivo canônico:", OUTPUT_CANONICO.resolve())
print("Dicionário:", DICTIONARY_PATH.resolve())
print("Relatório de qualidade:", QUALITY_PATH.resolve())
print()
print("Linhas:", len(dataset))
print("Colunas:", dataset.shape[1])
print("Contratos:", dataset["id_contrato"].nunique())
print(
    "Snapshots aptos para treino:",
    int(dataset["flag_apto_treino"].sum()),
)
display(quality)


DATASET DEFINITIVO GERADO
Arquivo definitivo: C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras\data\processed\dataset_modelagem_obras_snapshots_definitivo.csv
Arquivo canônico: C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras\data\processed\dataset_modelagem_obras_snapshots.csv
Dicionário: C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras\data\processed\dicionario_dataset_modelagem_definitivo.csv
Relatório de qualidade: C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras\reports\quality\qualidade_dataset_modelagem_definitivo.csv

Linhas: 587
Colunas: 80
Contratos: 45
Snapshots aptos para treino: 542


,metrica,valor
0,contratos_depurados,45
1,snapshots,587
2,snapshots_aptos_treino,542
3,contratos_aptos_treino,45
4,contratos_com_atraso,17
5,contratos_sem_atraso,28
6,taxa_atraso_contratos,0.3778
7,mediana_target_dias_contratos,0.0000
8,media_target_dias_contratos,20.6000
9,data_conclusao_min,2019-12-02 00:00:00



## 9. Campos que não devem entrar em X

Nunca usar diretamente como feature:

- `id_contrato`
- `data_corte`
- `status_final_controle`
- `data_conclusao_definitiva`
- `data_conclusao_controle`
- `data_termino_vigente_final_controle`
- `diferenca_vigente_original_final_controle`
- `target_dias_atraso`
- `target_atraso`
- `flag_apto_treino`
- `sample_weight_contrato_raw`
- `sample_weight_contrato`
- `split_contrato_temporal`
- `split_modelagem`

O peso deve ser passado ao `fit`, e o split deve ser usado apenas para separar treino, validação e teste.
